# Kronos daily +5% screener backtest — Production refit epoch 4 — 5D

Frozen Kronos checkpoint: **Production refit epoch 4**
Evaluation horizons: **Day 1 through Day 5**

For every rolling origin in the last 42 eligible sessions, the model
uses only the previous 120 sessions as context. Each horizon filters
positive predicted close gain, keeps at most 100 candidates, then
Optuna tunes one global rank-weight vector to select 30 stocks.

A hit means actual target-day high is at least 5% above the actual
previous trading-session close. Kronos weights are never updated.

## 1. Clone ISTL and pull only this model checkpoint from Git LFS

In [ ]:
from pathlib import Path
import os, subprocess

REPO = Path("/kaggle/working/ISTL")
if not REPO.exists():
    env = os.environ.copy()
    env["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(
        ["git", "clone", "https://github.com/zzeiidann/ISTL.git", str(REPO)],
        check=True,
        env=env,
    )
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)

subprocess.run(
    ["git", "-C", str(REPO), "lfs", "pull", "--include=Kronos IDX FineTune/results/2026-07-30/refit-run-e4/production_model/model.safetensors"],
    check=True,
)
checkpoint = REPO / "Kronos IDX FineTune/results/2026-07-30/refit-run-e4/production_model"
assert (checkpoint / "model.safetensors").exists(), checkpoint
print("Repository:", REPO)
print("Checkpoint:", checkpoint)

## 2. Install P100/T4-compatible runtime

In [ ]:
# CUDA 11.8 supports Kaggle Tesla P100 (sm_60) and T4 (sm_75).
%pip install -q --upgrade torch==2.3.1 --index-url https://download.pytorch.org/whl/cu118
%pip install -q einops==0.8.1 huggingface_hub==0.33.1 safetensors==0.6.2 pyarrow optuna==4.4.0 tqdm

## 3. Run rolling inference and Optuna 1,500-trial optimization

In [ ]:
import subprocess, sys

command = [
    sys.executable,
    str(REPO / "BackTest Kronos Screener" / "run_backtest.py"),
    "--repo", str(REPO),
    "--model-path", str(checkpoint),
    "--run-name", "production_refit_e4_5d",
    "--horizon-mode", "5",
    "--trials", "1500",
    "--backtest-sessions", "42",
    "--paths", "5",
    "--batch-size", "32",
    "--top-positive", "100",
    "--select", "30",
    "--min-universe-ratio", "0.80",
    "--lookback", "120",
    "--seed", "42",
]
print("Running:", " ".join(command))
subprocess.run(command, check=True)

## 4. Inspect and package outputs

In [ ]:
import json, shutil
import pandas as pd
from IPython.display import Javascript, display

output_dir = Path("/kaggle/working/backtest_kronos_screener/production_refit_e4_5d")
summary = json.loads((output_dir / "backtest_summary.json").read_text())
weights = json.loads((output_dir / "best_weights.json").read_text())
metrics = pd.read_csv(output_dir / "win_rate_by_horizon.csv")

print(json.dumps(summary, indent=2))
display(metrics)
display(
    pd.Series(weights, name="weight")
    .sort_values(key=abs, ascending=False)
    .rename_axis("feature")
    .to_frame()
)

archive = shutil.make_archive(
    f"/kaggle/working/production_refit_e4_5d", "zip", output_dir
)
print("Download:", archive)

# Start downloading automatically as soon as the ZIP exists.
archive_name = Path(archive).name
download_url = f"/files/kaggle/working/{archive_name}"
display(Javascript(
    "const link=document.createElement('a');"
    f"link.href={json.dumps(download_url)};"
    f"link.download={json.dumps(archive_name)};"
    "document.body.appendChild(link);"
    "link.click();"
    "link.remove();"
))